# 🔁 Rehacer solo algunos audios

Este cuaderno vuelve a generar **solo los audios que fallaron** (6 tramos), con tu misma voz.

1. **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar.**
2. **Entorno de ejecución → Ejecutar todo.** La primera vez la sesión se reinicia sola después de instalar: vuelve a **Ejecutar todo**.
3. Al final se descarga `audios_corregidos.zip` (tarda 2–3 minutos después de instalar). Súbelo al chat con Claude.

## 1. Revisar que haya GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("❌ No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4, y vuelve a ejecutar todo.")
print("✅ GPU lista:", r.stdout.strip())

## 2. Instalar el modelo (tarda 2–4 minutos)
La primera vez, al terminar la instalación **la sesión se reinicia sola** (es normal). Entonces vuelve a usar **Entorno de ejecución → Ejecutar todo**: la segunda vez la instalación se salta y sigue directo.

In [ ]:
import importlib.metadata as md, os, subprocess
def version(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None
sh = lambda cmd: subprocess.run(cmd, shell=True).returncode
reiniciar = False
if version("chatterbox-tts") is None:
    sh("pip install -q chatterbox-tts")
    reiniciar = True
# chatterbox usa torch 2.6.0: torchvision debe ser la versión compatible (0.21.0)
tv = version("torchvision")
if tv is not None and not tv.startswith("0.21"):
    if sh("pip install -q torchvision==0.21.0") != 0:
        sh("pip uninstall -y -q torchvision")
    reiniciar = True
if reiniciar:
    print("✅ Instalación lista. La sesión se reiniciará sola para aplicar los cambios.")
    print("👉 Cuando aparezca el aviso, vuelve a: Entorno de ejecución → Ejecutar todo.")
    os.kill(os.getpid(), 9)
else:
    print("✅ Todo instalado y compatible")

## 3. Tu voz de referencia
Ya está cargada automáticamente desde tu repositorio (tomada de tu video). **No necesitas subir nada.** Si algún día quieres usar otra grabación, cambia `SUBIR_OTRA = False` por `True`.

In [ ]:
import os, subprocess, urllib.request
SUBIR_OTRA = False
if SUBIR_OTRA:
    from google.colab import files
    subido = files.upload(); nombre = list(subido.keys())[0]
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",nombre,"-af","silenceremove=start_periods=1:start_threshold=-45dB",
                    "-ac","1","-ar","24000","-t","25","voz_ref.wav"], check=True)
elif not os.path.exists("voz_ref.wav"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/profcristobalespinoza-netizen/m1-clase-1/main/voz/voz_ref.wav", "voz_ref.wav")
print("✅ Audio de referencia listo")
from IPython.display import Audio, display
display(Audio("voz_ref.wav"))

## 4. Cargar el modelo y los guiones

In [ ]:
import json, urllib.request, torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
try:
    model = ChatterboxMultilingualTTS.from_pretrained(device="cuda", t3_model="v3")
except TypeError:
    model = ChatterboxMultilingualTTS.from_pretrained(device="cuda")
GUIONES = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/profcristobalespinoza-netizen/m1-clase-1/main/voz/guiones.json"))
print("✅ Modelo cargado ·", len(GUIONES), "ejercicios")

import re
def frases(texto):
    partes = re.split(r'(?<=[.!?¿¡:])\s+', texto.strip())
    out, buf = [], ""
    for p in partes:
        if len(buf) + len(p) < 220: buf = (buf + " " + p).strip()
        else:
            if buf: out.append(buf)
            buf = p
    if buf: out.append(buf)
    return out

def narrar(texto, destino):
    trozos = []
    pausa = torch.zeros(1, int(model.sr * 0.30))
    for f in frases(texto):
        wav = model.generate(f, language_id="es", audio_prompt_path="voz_ref.wav")
        trozos += [wav.cpu(), pausa]
    ta.save(destino, torch.cat(trozos, dim=1), model.sr)

## 5. Rehacer los audios y descargar

In [ ]:
import os, shutil
from IPython.display import Audio, display
REHACER = ['08_05', '27_05', '34_05', '24_04', '26_03', '33_01']
os.makedirs("corregidos", exist_ok=True)
for clave in REHACER:
    n, k = clave.split("_")
    texto = GUIONES[str(int(n))][int(k)]
    destino = f"corregidos/ej{clave}.wav"
    narrar(texto, destino)
    print("✅", clave, "·", texto)
    display(Audio(destino))
shutil.make_archive("audios_corregidos", "zip", "corregidos")
from google.colab import files
files.download("audios_corregidos.zip")